# Run Part A on Google Colab

This notebook runs the Part A SAM vs SGD performance experiment on Google Colab. It is meant to be opened in Colab, connected to a GPU runtime, and used as the shared running guide for the group.

Before running the training cell, use `Runtime > Change runtime type > Hardware accelerator > GPU`.

## 1. Mount Google Drive

The repository and generated checkpoints/results will be stored in Google Drive so they persist after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update the GitHub repository

If the folder already exists in Drive, this cell pulls the latest code from GitHub. If it does not exist, it clones the repository.

In [ ]:
import os

REPO_URL = "https://github.com/666junyichen/Understanding-SAM-s-Generalization-through-Effective-Dimensionality.git"
PROJECT_ROOT = "/content/drive/MyDrive/Assignment2_SAM"

if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    %cd {PROJECT_ROOT}
    !git pull

%cd {PROJECT_ROOT}
!ls

## 3. Check GPU availability

If CUDA is not available, change the runtime type to GPU before running the full experiment.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")

## 4. Install dependencies

Colab usually already has PyTorch installed. This cell installs the packages used by the project.

In [ ]:
!pip install -r requirements.txt

## 5. Run tests

This checks that the dataloader, model, SAM optimizer, training loop, evaluation, plotting helpers, and experiment entry point import correctly.

In [ ]:
!python -m unittest discover -s tests -v

## 6. Confirm or edit experiment settings

The formal setting should be `EPOCHS = 50` and `DEBUG_SUBSET_SIZE = None` in `config.py`. For a quick smoke test, temporarily use `EPOCHS = 1` and `DEBUG_SUBSET_SIZE = 512`.

In [ ]:
!grep -n "EPOCHS\|DEBUG_SUBSET_SIZE\|BATCH_SIZE\|NUM_WORKERS" config.py

Optional quick edit for smoke testing. Do not run this cell for final results.

In [ ]:
# Optional smoke-test settings. Uncomment only if you want a quick test run.
# !sed -i 's/^EPOCHS = .*/EPOCHS = 1/' config.py
# !sed -i 's/^DEBUG_SUBSET_SIZE = .*/DEBUG_SUBSET_SIZE = 512/' config.py
# !grep -n "EPOCHS\|DEBUG_SUBSET_SIZE" config.py

## 7. Run the full Part A experiment

This trains SGD and SAM-SGD from the same initialization, evaluates ID and OOD performance at every epoch, saves `last`, `best_id`, and `best_ood` checkpoints, and writes CSV metrics. SAM uses two forward/backward passes per batch, so it is slower than SGD.

In [ ]:
!python run_experiment.py

## 8. Generate plots and summary table

This creates the nine Part A report-ready figures and the summary table. The OOD trajectory plot requires the updated per-epoch OOD metrics written by `run_experiment.py`.

In [ ]:
!python plot_results.py

## 9. Inspect outputs

The checkpoint files store the trained SGD and SAM-SGD models. The CSV files in `results/` and PNG files in `figures/` are useful for the Part A report.

In [ ]:
!ls -lh checkpoints
!ls -lh results
!ls -lh figures

import pandas as pd

display(pd.read_csv('results/metrics.csv'))
display(pd.read_csv('results/summary_table.csv'))

## 10. Optional: load a selected checkpoint

Use this pattern to verify that a trained SGD or SAM-SGD checkpoint can be loaded correctly. Choose `last`, `best_id`, or `best_ood` depending on which Part A result you want to inspect.

In [ ]:
from models import get_model
from utils import load_checkpoint, get_device

device = get_device()
model = get_model('resnet18').to(device)

# Options:
# checkpoints/sgd_resnet18_cifar10_last.pt
# checkpoints/sgd_resnet18_cifar10_best_id.pt
# checkpoints/sgd_resnet18_cifar10_best_ood.pt
# checkpoints/sam_resnet18_cifar10_last.pt
# checkpoints/sam_resnet18_cifar10_best_id.pt
# checkpoints/sam_resnet18_cifar10_best_ood.pt
checkpoint_path = 'checkpoints/sam_resnet18_cifar10_best_ood.pt'
checkpoint = load_checkpoint(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Loaded:', checkpoint['optimizer_name'], checkpoint['model_name'])
print('Checkpoint type:', checkpoint['checkpoint_type'])
print('Epoch:', checkpoint['epoch'])
print('Metrics:', checkpoint['metrics'])

## Notes for sharing

- GitHub stores the code, but generated datasets, checkpoints, CSV files, and PNG figures are ignored by Git.
- Share the Google Drive folder or copy the `checkpoints/`, `results/`, and `figures/` folders to a shared group folder after training.
- Keep source code and notebooks in GitHub.
- Share `.pt` checkpoints through Google Drive instead of GitHub.
- Do not upload CIFAR-10 data, zip files, or large raw generated files.